# Parte Específica: valores "Unknown" no BankChurners

No BankChurners a ausência não aparece como `NaN`: ela foi registrada como o texto `"Unknown"`
em três variáveis categóricas (`Education_Level`, `Marital_Status`, `Income_Category`).

Este notebook investiga se a ausência depende de outras variáveis (para apoiar a discussão do
mecanismo de ausência) e aplica o tratamento escolhido pelo grupo.

In [ ]:
# [IA] Claude Code – importacao das bibliotecas e leitura do dataset
from pathlib import Path

import pandas as pd

CAMINHO_DADOS = Path("../../data/parte_especifica/BankChurners.csv")
df = pd.read_csv(CAMINHO_DADOS)

COLUNAS_COM_UNKNOWN = ["Education_Level", "Marital_Status", "Income_Category"]

## 1. Quantidade de Unknown

In [ ]:
# [IA] Claude Code – quantidade e porcentagem de Unknown em cada coluna
resumo_unknown = pd.DataFrame({
    "qtd_unknown": [(df[c] == "Unknown").sum() for c in COLUNAS_COM_UNKNOWN],
}, index=COLUNAS_COM_UNKNOWN)
resumo_unknown["pct_unknown"] = (resumo_unknown["qtd_unknown"] / len(df) * 100).round(2)
resumo_unknown

In [ ]:
# [IA] Claude Code – colunas indicadoras: 1 se o valor e Unknown, 0 se e conhecido
indicadores = pd.DataFrame({
    f"unknown_{c}": (df[c] == "Unknown").astype(int) for c in COLUNAS_COM_UNKNOWN
})
indicadores.sum()

## 2. Os Unknown aparecem juntos?

In [ ]:
# [IA] Claude Code – combinacoes de Unknown nas tres colunas (1 = Unknown)
indicadores.value_counts().rename("qtd_linhas").reset_index()

In [ ]:
# [IA] Claude Code – taxa de Unknown em cada coluna, separando quem tem ou nao Unknown nas outras
linhas = []
for alvo in COLUNAS_COM_UNKNOWN:
    for outra in COLUNAS_COM_UNKNOWN:
        if outra == alvo:
            continue
        grupo = indicadores[f"unknown_{outra}"]
        taxa = indicadores[f"unknown_{alvo}"].groupby(grupo).mean() * 100
        linhas.append({
            "coluna": alvo,
            "condicao": f"{outra} e Unknown?",
            "pct_unknown_quando_nao": round(taxa.get(0, float("nan")), 2),
            "pct_unknown_quando_sim": round(taxa.get(1, float("nan")), 2),
        })
pd.DataFrame(linhas)

## 3. A taxa de Unknown muda conforme outras variáveis categóricas?

In [ ]:
# [IA] Claude Code – % de Unknown em cada coluna, por categoria de outras variaveis
def taxa_unknown_por(variavel):
    tabela = pd.concat([df[variavel], indicadores], axis=1).groupby(variavel).mean() * 100
    tabela.columns = [c.replace("unknown_", "pct_unknown_") for c in tabela.columns]
    tabela["qtd_linhas"] = df[variavel].value_counts()
    return tabela.round(2)

taxa_unknown_por("Attrition_Flag")

In [ ]:
# [IA] Claude Code – mesma tabela por genero
taxa_unknown_por("Gender")

In [ ]:
# [IA] Claude Code – mesma tabela por categoria do cartao
taxa_unknown_por("Card_Category")

## 4. Quem tem Unknown é diferente nas variáveis numéricas?

In [ ]:
# [IA] Claude Code – mediana das variaveis numericas: linhas com Unknown x linhas com valor conhecido
colunas_numericas = df.select_dtypes(include="number").columns.drop("CLIENTNUM")

comparacoes = {}
for c in COLUNAS_COM_UNKNOWN:
    eh_unknown = df[c] == "Unknown"
    comparacoes[(c, "conhecido")] = df.loc[~eh_unknown, colunas_numericas].median()
    comparacoes[(c, "unknown")] = df.loc[eh_unknown, colunas_numericas].median()

medianas = pd.DataFrame(comparacoes).round(2)
medianas

In [ ]:
# [IA] Claude Code – diferenca relativa (%) da mediana: (unknown - conhecido) / conhecido
diferenca_pct = pd.DataFrame({
    c: ((medianas[(c, "unknown")] - medianas[(c, "conhecido")]) / medianas[(c, "conhecido")] * 100)
    for c in COLUNAS_COM_UNKNOWN
}).round(1)
diferenca_pct

## 5. Mecanismo de ausência

Para Education_Level, o mecanismo mais próximo parece ser MCAR, pois as taxas de Unknown apresentam poucas diferenças entre os grupos analisados. Para Marital_Status, existem algumas diferenças conforme outras variáveis, então podemos considerar a possibilidade de MAR. Já em Income_Category, a ausência apresenta uma diferença muito grande entre homens e mulheres, indicando um comportamento mais próximo de MAR. Não é possível afirmar esses mecanismos com certeza apenas pelas tabelas, principalmente porque não conseguimos observar diretamente o motivo pelo qual cada cliente deixou de informar seus dados.


## 6. Tratamento

In [ ]:
# [IA] Claude Code – aplicar o tratamento escolhido pelo grupo
# Opcoes:
#   "categoria_propria": mantem "Unknown" como uma categoria a parte (nenhuma linha e removida
#                        e nenhum valor e inventado)
#   "moda":              substitui "Unknown" pela categoria mais frequente da coluna
# O enunciado nao permite simplesmente remover as linhas com ausencia.
TRATAMENTO = "categoria_propria"  # (decisao do grupo)

df_tratado = df.copy()
if TRATAMENTO == "categoria_propria":
    pass
elif TRATAMENTO == "moda":
    for c in COLUNAS_COM_UNKNOWN:
        moda = df.loc[df[c] != "Unknown", c].mode()[0]
        df_tratado[c] = df_tratado[c].replace("Unknown", moda)
else:
    raise ValueError(f"Tratamento desconhecido: {TRATAMENTO}")

# Conferencia: nenhuma linha foi removida
assert len(df_tratado) == len(df)
pd.DataFrame({c: df_tratado[c].value_counts() for c in COLUNAS_COM_UNKNOWN}).fillna(0).astype(int)

In [ ]:
# [IA] Claude Code – salvar o dataset tratado para as proximas etapas (EDA e dashboard)
PASTA_SAIDA = Path("../../out/parte_especifica")
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
df_tratado.to_csv(PASTA_SAIDA / "bankchurners_tratado.csv", index=False)
print("Salvo em", PASTA_SAIDA / "bankchurners_tratado.csv")

### Justificativa do tratamento

Decidimos manter Unknown como uma categoria própria porque não sabemos qual seria o valor correto de cada informação ausente. Substituir pela moda poderia criar informações que não correspondem à realidade dos clientes. Além disso, manter essa categoria permite analisar os registros sem excluir clientes ou perder informações importantes.
